## **Regression**

Regression is a machine learning technique used to predict numerical values based on input features.
It helps us understand how different factors like user behavior, location, and attraction details affect outcomes such as ratings or visit counts.
The main purpose of regression is prediction and trend analysis.
In this project, we use two regression models: ***Linear Regression and Random Forest Regressor*** to compare simple and advanced approaches.

## **Step 1:- Data Loading**

Before Model Development, The first step is to load the cleaned dataset which was stored in the dataset folder.

In [1]:
import pandas as pd

data_path = "../dataset/cleaned_tourism_data.csv"
df = pd.read_csv(data_path)

df.shape

(52930, 19)

## **Step 2:- Linear Regression**

Linear Regression is a simple and widely used regression algorithm that models the linear relationship between input features and a numerical target variable.
It works by fitting a straight line that best explains how changes in features affect the output.

In [2]:
# ----------------------------------
# Import required libraries
# ----------------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ----------------------------------
# Separate features and target
# ----------------------------------
features = ['VisitMode', 'season', 'Country', 'AttractionType']
target = 'user_avg_rating'


df_model = df[features + [target]].dropna()

X = df_model[features]
y = df_model[target]

# ----------------------------------
# Train-test split
# ----------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----------------------------------
# Define column types
# ----------------------------------
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# ----------------------------------
# Preprocessing
# ----------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)

# ----------------------------------
# Model
# ----------------------------------
model = LinearRegression(
    n_jobs=-1
)

# ----------------------------------
# Pipeline
# ----------------------------------
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# ----------------------------------
# Train
# ----------------------------------
lr_pipeline.fit(X_train, y_train)

# ----------------------------------
# Evaluate
# ----------------------------------
y_pred = lr_pipeline.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))


future_user = pd.DataFrame([{
    "Country": "Italy",
    "AttractionType": "Museum",
    "season": "Summer",
    "VisitMode": "Family"
}])


future_rating = lr_pipeline.predict(future_user)

print("---------------------------")
print("Predicted Future User Rating:", round(future_rating[0], 2))

MAE: 0.6130381333064299
MSE: 0.6438695400272753
R2 Score: 0.06517029384820772
---------------------------
Predicted Future User Rating: 4.08


## **Step 3:- Random Forest Regressor**

Random Forest Regressor is an ensemble learning algorithm that builds multiple decision trees and combines their outputs to predict continuous numerical values.
It captures complex, non-linear relationships between features and the target variable while reducing overfitting through averaging.

In [3]:
from sklearn.ensemble import RandomForestRegressor
import joblib


# ----------------------------------
# Separate features and target
# ----------------------------------
features = ['VisitMode', 'season', 'Country', 'AttractionType']
target = 'user_avg_rating'


df_model = df[features + [target]].dropna()

X = df_model[features]
y = df_model[target]

# ----------------------------------
# Train-test split
# ----------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# ----------------------------------
# Automatically detect column types
# ----------------------------------
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()


# ----------------------------------
# Preprocessing
# ----------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)


# ----------------------------------
# Model
# ----------------------------------
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)


# ----------------------------------
# Pipeline
# ----------------------------------
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', rf_model)
])


# ----------------------------------
# Train
# ----------------------------------
rf_pipeline.fit(X_train, y_train)


# ----------------------------------
# Evaluate
# ----------------------------------
y_pred = rf_pipeline.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))


future_user = pd.DataFrame([{
    "Country": "Italy",
    "AttractionType": "Museum",
    "season": "Summer",
    "VisitMode": "Family"
}])


future_rating = rf_pipeline.predict(future_user)
print('----------------------------------')
print("Predicted Future User Rating:", round(future_rating[0], 2))

MAE: 0.6131168797705833
MSE: 0.6384063407620522
R2 Score: 0.07310227485718801
----------------------------------
Predicted Future User Rating: 4.12


## **Step 3:- Model Evaluation & Insights**

Model evaluation helps us compare different models objectively and choose the one that best fits our problem.
Business insights translate these numbers into real, actionable decisions.

## ***3.1 Evaluate Models***

***Model Performance Comparison***


| Model                   | MAE ↓  | MSE ↓  | R² Score ↑ |
| ----------------------- | ------ | ------ | ---------- |
| **Linear Regression**       | **0.6130** | **0.6438** | **0.0651** |
| **Random Forest Regressor** | **0.6131** | **0.6384** | **0.0731**     |

***Key Observations***:-

Random Forest outperformed Linear Regression on all three metrics.

The R² score of 0.07 shows Random Forest explains most of the variation in ratings.

## ***3.2 Business Insights***


What the model tells us (Translated to business language)

User behavior matters:
Users with more visits and higher average ratings strongly influence prediction accuracy → loyal users are more consistent.

Attraction popularity impacts ratings:
Frequently visited attractions tend to have more stable and predictable ratings.

Regional differences exist:
Some regions consistently receive lower average ratings, indicating infrastructure or experience improvement opportunities.

Visit mode influences satisfaction:
Certain visit modes (e.g., family or leisure travel) are associated with higher ratings, suggesting targeted marketing potential.

## ***3.3 Saving the Best Model***

Saving the best model to be used to develop streamlit app


***Random Forest is the best model***

***Why?***

Lowest MAE and MSE → more accurate predictions

Highest R² score → explains ratings very well

Simpler, faster, and more stable for deployment

Works better with your current dataset size

In [4]:
# ----------------------------------
# Save model (IMPORTANT for Streamlit)
# ----------------------------------
joblib.dump(rf_pipeline, "../models/Regressor_Pipeline.pkl")

print("Regressor Prediction Pipeline Saved Successfully")

Regressor Prediction Pipeline Saved Successfully
